In [1]:
import pandas as pd
df = pd.read_csv(r'D:\Enterprise_ITSM_AI\data\raw\itsm_cleaned_dataset.csv')

In [2]:
df['made_sla'].value_counts()

made_sla
True     132497
False      9215
Name: count, dtype: int64

In [3]:
# Features
X = df[
    [
        "contact_type",
        "location",
        "u_symptom",
        "impact",
        "urgency",
        "knowledge",
        "notify",
        "opened_hour",
        "opened_day_of_week",
        "opened_month",
        "is_weekend"
    ]
]

# Target
y = df["made_sla"]

In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

In [9]:
preprocessing_data = ColumnTransformer(
    transformers=[
        (
            "trf1",
            OrdinalEncoder(
                categories=[
                    ["1 - High", "2 - Medium", "3 - Low"],
                    ["1 - High", "2 - Medium", "3 - Low"]
                ]
            ),
            ["impact", "urgency"]
        ),
        (
            "trf2",
            OneHotEncoder(handle_unknown="ignore"),
            [
                "contact_type",
                "location",
                "u_symptom",
                "notify",
                "opened_day_of_week"
            ]
        )
    ],
    remainder="passthrough" 
)
dt_pipeline = Pipeline([
    ("preprocessor", preprocessing_data),
    ("model", DecisionTreeClassifier(random_state=42))
])
rf_pipeline = Pipeline([
    ("preprocessor", preprocessing_data),
    ("model", RandomForestClassifier(
        n_estimators=300,
        max_depth=15,
        min_samples_split=5,
        min_samples_leaf=2,
        max_features="sqrt",
        random_state=42,
        n_jobs=-1
    ))
])
xgboost_pipeline = Pipeline([
    ("preprocessor", preprocessing_data),
    ("model", XGBClassifier(
        random_state=42,
        eval_metric="logloss"
    ))
])

In [10]:
dt_pipeline.fit(X_train, y_train)

rf_pipeline.fit(X_train, y_train)

xgboost_pipeline.fit(X_train, y_train)

c:\Users\bharg\anaconda3\Lib\site-packages\sklearn\compose\_column_transformer.py:1623: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('trf1',
                                                  OrdinalEncoder(categories=[['1 '
                                                                              '- '
                                                                              'High',
                                                                              '2 '
                                                                              '- '
                                                                              'Medium',
                                                                              '3 '
                                                                              '- '
                                                                              'Low'],
                                                                             ['1 '
                                                                              '- '
                                                                              'High',
                                                                              '2 '
                                                                              '- '
                                                                              'Medium',
                                                                              '3 '
                                                                              '- '
                                                                              'Low']]),
                                                  ['impact', 'urgency']),
                                                 ('trf2',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['contact_type', 'location',
                                                   'u_symptom', 'notify',
                                                   'opened_day_of_week'])])...
                               feature_types=None, feature_weights=None,
                               gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=None,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=None, max_leaves=None,
                               min_child_weight=None, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=None, n_jobs=None,
                               num_parallel_tree=None, ...))])

In [15]:
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score

In [ ]:
results={}
def evaluate_model(name,model,X_test,y_test):
    y_pred=model.predict(X_test)
    metrics={
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, average="macro"),
        "Recall": recall_score(y_test, y_pred, average="macro"),
        "F1 Score": f1_score(y_test, y_pred, average="macro")
    }
    results[name]=metrics

In [17]:
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score

In [18]:
evaluate_model("Decision Tree", dt_pipeline, X_test, y_test)
evaluate_model("Random Forest", rf_pipeline, X_test, y_test)
evaluate_model("XGBoost", xgboost_pipeline, X_test, y_test)

c:\Users\bharg\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [19]:
results_df=pd.DataFrame(results).T
results_df = results_df.round(4)
print(results_df)

               Accuracy  Precision  Recall  F1 Score
Decision Tree    0.9225     0.4844  0.4966    0.4857
Random Forest    0.9350     0.4675  0.5000    0.4832
XGBoost          0.9349     0.4675  0.5000    0.4832


In [20]:
import joblib

joblib.dump(
    dt_pipeline,
    r"D:\Enterprise_ITSM_AI\models_saved\sla_prediction_pipeline.joblib"
)

['D:\\Enterprise_ITSM_AI\\models_saved\\sla_prediction_pipeline.joblib']

In [21]:
import joblib

joblib.dump(
    dt_pipeline,
    r"D:\Enterprise_ITSM_AI\models_saved\sla_prediction_pipeline.joblib"
)

['D:\\Enterprise_ITSM_AI\\models_saved\\sla_prediction_pipeline.joblib']